# 02. RFM 분석 + K-Means 고객 세그멘테이션

UCI Online Retail 데이터로 RFM(Recency, Frequency, Monetary) 지표를 계산하고,
K-Means 클러스터링으로 고객을 4개 세그먼트로 분류합니다.

**분석 목표**:
1. RFM 지표 엔지니어링 (고객별 최근성, 빈도, 금액)
2. K-Means 최적 k 결정 (Elbow Method + Silhouette Score)
3. 세그먼트 프로파일링 및 비즈니스 전략 도출
4. 통계적 검증 (ANOVA, Silhouette)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11

print('Libraries loaded.')

## 1. 데이터 로딩 및 정제

In [ ]:
df = pd.read_csv('../data/online_retail.csv', encoding='utf-8')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# 컬럼명 통일
if 'Customer ID' in df.columns:
    df.rename(columns={'Customer ID': 'CustomerID'}, inplace=True)

price_col = 'Price' if 'Price' in df.columns else 'UnitPrice'
invoice_col = 'Invoice' if 'Invoice' in df.columns else 'InvoiceNo'

df['Revenue'] = df['Quantity'] * df[price_col]

# 정제: 반품 제외, 양수 금액, CustomerID 존재
df_clean = df[
    (df['Quantity'] > 0) &
    (df[price_col] > 0) &
    (df['CustomerID'].notna())
].copy()

print(f'정제 데이터: {len(df_clean):,} rows, {df_clean["CustomerID"].nunique():,} customers')

## 2. RFM Feature Engineering

In [ ]:
# 기준일: 데이터 최종일 + 1
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f'Snapshot date: {snapshot_date}')

# RFM 계산
rfm = df_clean.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=(invoice_col, 'nunique'),
    Monetary=('Revenue', 'sum')
).reset_index()

print(f'\nRFM Table: {len(rfm):,} customers')
rfm.describe()

In [ ]:
# RFM 분포 시각화
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, color in zip(axes, ['Recency', 'Frequency', 'Monetary'],
                            ['#4e79a7', '#f28e2b', '#59a14f']):
    # 99th percentile로 clipping (시각화용)
    data = rfm[col].clip(upper=rfm[col].quantile(0.99))
    ax.hist(data, bins=40, color=color, edgecolor='white', alpha=0.8)
    ax.axvline(rfm[col].median(), color='#e15759', linestyle='--',
               label=f'Median: {rfm[col].median():.0f}')
    ax.set_title(f'{col} Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel(col)
    ax.legend()

plt.suptitle('RFM Feature Distributions', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. RFM Score (Quintile-based)

In [ ]:
# Quintile (1-5) 스코어링
# Recency는 낮을수록 좋으므로 역순
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

print('=== RFM Score Distribution ===')
print(rfm['RFM_Score'].describe())
print(f'\nScore Range: {rfm["RFM_Score"].min()} ~ {rfm["RFM_Score"].max()}')

rfm.head(10)

## 4. K-Means Clustering

### 4-1. Elbow Method + Silhouette Score로 최적 k 결정

In [ ]:
# 스케일링 (K-Means는 거리 기반이므로 스케일링 필수)
features = rfm[['Recency', 'Frequency', 'Monetary']].copy()

# Log 변환 (왜도가 높은 Frequency, Monetary)
features['Frequency'] = np.log1p(features['Frequency'])
features['Monetary'] = np.log1p(features['Monetary'])

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Elbow Method + Silhouette
K_range = range(2, 9)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(features_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(features_scaled, labels))
    print(f'k={k}: Inertia={km.inertia_:.0f}, Silhouette={silhouettes[-1]:.4f}')

best_k = list(K_range)[np.argmax(silhouettes)]
print(f'\nBest k by Silhouette: {best_k} (score={max(silhouettes):.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow Curve
axes[0].plot(list(K_range), inertias, marker='o', linewidth=2, color='#4e79a7')
axes[0].axvline(4, color='#e15759', linestyle='--', alpha=0.7, label='k=4')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (Within-cluster SS)')
axes[0].set_title('Elbow Method', fontsize=14, fontweight='bold')
axes[0].legend()

# Silhouette Score
axes[1].plot(list(K_range), silhouettes, marker='s', linewidth=2, color='#59a14f')
axes[1].axvline(best_k, color='#e15759', linestyle='--', alpha=0.7, label=f'Best k={best_k}')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score by k', fontsize=14, fontweight='bold')
axes[1].legend()

plt.suptitle('Optimal k Selection', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 4-2. K-Means (k=4) 적용

In [ ]:
K = 4
km = KMeans(n_clusters=K, random_state=42, n_init=10)
rfm['Cluster'] = km.fit_predict(features_scaled)

# Silhouette score
sil_score = silhouette_score(features_scaled, rfm['Cluster'])
print(f'K-Means (k={K}) Silhouette Score: {sil_score:.4f}')

# 클러스터별 RFM 평균
cluster_profile = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].agg(['mean', 'median', 'count'])
print('\n=== Cluster Profiles ===')
cluster_profile

In [ ]:
# 클러스터별 RFM 평균으로 세그먼트 명명
cluster_means = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean()

# 세그먼트 레이블 자동 할당 (RFM 조합)
# 낮은 Recency + 높은 Frequency + 높은 Monetary → Champions
# 낮은 Recency + 낮은 Frequency → New/Potential
# 높은 Recency + 높은 Frequency → At-Risk
# 높은 Recency + 낮은 Frequency + 낮은 Monetary → Lost

segment_names = {}
sorted_clusters = cluster_means.sort_values('Monetary', ascending=False).index

labels_pool = ['Champions', 'Loyal Customers', 'At-Risk', 'Lost']

# Recency 기준 정렬: 낮은 Recency(최근) → Champions/Loyal, 높은 Recency → At-Risk/Lost
recency_sorted = cluster_means.sort_values('Recency')
for i, (cluster_id, row) in enumerate(recency_sorted.iterrows()):
    if i < 2:  # 최근 방문 그룹
        if row['Monetary'] > cluster_means['Monetary'].median():
            segment_names[cluster_id] = 'Champions'
        else:
            segment_names[cluster_id] = 'Loyal Customers'
    else:  # 오래된 방문 그룹
        if row['Monetary'] > cluster_means['Monetary'].median():
            segment_names[cluster_id] = 'At-Risk'
        else:
            segment_names[cluster_id] = 'Lost'

rfm['Segment'] = rfm['Cluster'].map(segment_names)

print('=== Segment Assignment ===')
for cid, name in segment_names.items():
    row = cluster_means.loc[cid]
    count = (rfm['Cluster'] == cid).sum()
    print(f'Cluster {cid} → {name}: R={row["Recency"]:.0f}d, F={row["Frequency"]:.1f}, M=£{row["Monetary"]:,.0f} ({count:,} customers)')

## 5. 세그먼트 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

segment_colors = {'Champions': '#59a14f', 'Loyal Customers': '#4e79a7',
                  'At-Risk': '#f28e2b', 'Lost': '#e15759'}

# Panel 1: Segment size (pie)
seg_counts = rfm['Segment'].value_counts()
colors_ordered = [segment_colors[s] for s in seg_counts.index]
axes[0, 0].pie(seg_counts, labels=seg_counts.index, autopct='%1.1f%%',
               colors=colors_ordered, startangle=90,
               textprops={'fontsize': 10})
axes[0, 0].set_title('Customer Segment Distribution', fontsize=13, fontweight='bold')

# Panel 2: RFM Heatmap (normalized)
seg_rfm = rfm.groupby('Segment')[['Recency', 'Frequency', 'Monetary']].mean()
# 정규화: 각 RFM을 0-1 스케일로
seg_rfm_norm = seg_rfm.copy()
for col in seg_rfm_norm.columns:
    if col == 'Recency':  # Recency는 낮을수록 좋으므로 역전
        seg_rfm_norm[col] = 1 - (seg_rfm_norm[col] - seg_rfm_norm[col].min()) / (seg_rfm_norm[col].max() - seg_rfm_norm[col].min())
    else:
        seg_rfm_norm[col] = (seg_rfm_norm[col] - seg_rfm_norm[col].min()) / (seg_rfm_norm[col].max() - seg_rfm_norm[col].min())

sns.heatmap(seg_rfm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=axes[0, 1], cbar_kws={'label': 'Normalized Score'})
axes[0, 1].set_title('RFM Score by Segment (Normalized)', fontsize=13, fontweight='bold')

# Panel 3: Recency vs Monetary scatter
for seg, color in segment_colors.items():
    mask = rfm['Segment'] == seg
    subset = rfm[mask].sample(min(500, mask.sum()), random_state=42)  # 시각화용 샘플링
    axes[1, 0].scatter(subset['Recency'], np.log1p(subset['Monetary']),
                        alpha=0.5, s=20, c=color, label=seg)
axes[1, 0].set_xlabel('Recency (days)')
axes[1, 0].set_ylabel('log(Monetary + 1)')
axes[1, 0].set_title('Recency vs Monetary by Segment', fontsize=13, fontweight='bold')
axes[1, 0].legend(fontsize=9)

# Panel 4: Revenue share by segment
seg_revenue = rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False)
seg_revenue_pct = seg_revenue / seg_revenue.sum() * 100
colors_bar = [segment_colors[s] for s in seg_revenue.index]

bars = axes[1, 1].bar(seg_revenue.index, seg_revenue_pct, color=colors_bar)
axes[1, 1].set_ylabel('% of Total Revenue')
axes[1, 1].set_title('Revenue Contribution by Segment', fontsize=13, fontweight='bold')

for bar, pct in zip(bars, seg_revenue_pct):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{pct:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('K-Means Customer Segmentation (k=4)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. 통계적 검증: ANOVA

각 RFM 지표가 클러스터 간에 통계적으로 유의미하게 다른지 검증합니다.

In [ ]:
print('=== One-Way ANOVA: RFM Features Across Clusters ===\n')

for col in ['Recency', 'Frequency', 'Monetary']:
    groups = [group[col].values for _, group in rfm.groupby('Cluster')]
    f_stat, p_val = stats.f_oneway(*groups)

    # Effect size (eta-squared)
    grand_mean = rfm[col].mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
    ss_total = sum((rfm[col] - grand_mean)**2)
    eta_sq = ss_between / ss_total

    print(f'{col}:')
    print(f'  F-statistic: {f_stat:.2f}')
    print(f'  p-value: {p_val:.2e}')
    print(f'  Eta-squared (effect size): {eta_sq:.4f}')
    print(f'  → {"Highly significant" if p_val < 0.001 else "Significant" if p_val < 0.05 else "Not significant"}\n')

print(f'\nOverall Silhouette Score: {sil_score:.4f}')
print('→ 클러스터링이 RFM 공간에서 유의미한 분리를 달성했습니다.')

## 7. 비즈니스 추천

| 세그먼트 | 특성 | 전략 |
|----------|------|------|
| **Champions** | 최근 구매, 고빈도, 고금액 | VIP 대우, 얼리 액세스, 로열티 프로그램 |
| **Loyal Customers** | 꾸준한 구매, 중간 금액 | 업셀 기회, 카테고리 확장 유도 |
| **At-Risk** | 과거 고가치였으나 최근 미방문 | 재활성화 캠페인 (할인 쿠폰, 개인화 이메일) |
| **Lost** | 오래 전 저가치 구매 후 이탈 | 저비용 리타겟팅 또는 자연 이탈 수용 |

**핵심 인사이트**: Champions + Loyal 세그먼트가 전체 매출의 대부분을 차지.
→ At-Risk 세그먼트의 Champions 재전환이 가장 높은 ROI를 기대할 수 있습니다.

→ 다음 노트북(03_churn_eda)에서 이탈 예측을 위한 Telco 데이터 분석으로 이어집니다.